In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/dockehn restart ke baad r-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

In [ ]:
image_size = 224

train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/train.csv")
print(len(df))

In [ ]:
def get_image_path(image_id):
    return f"/kaggle/input/competitions/landmark-recognition-2021/train/{image_id[0]}/{image_id[1]}/{image_id[2]}/{image_id}.jpg"

In [ ]:
df = df.groupby('landmark_id').filter(lambda x: len(x) >= 10)

In [ ]:


top_classes = df['landmark_id'].value_counts().head(20).index
df = df[df['landmark_id'].isin(top_classes)].reset_index(drop=True)

In [ ]:
df = df.sample(min(3000,len(df))).reset_index(drop=True)

In [ ]:
df["image_path"] = df["id"].apply(get_image_path)

In [ ]:
print("unique landmark_ids:", df["landmark_id"].nunique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["landmark_id"])

In [ ]:
print("num_classes:", df["label_encoded"].nunique())
print("min:", df["label_encoded"].min())
print("max:", df["label_encoded"].max())

In [ ]:
# Step 3: split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    stratify=df["label_encoded"],
    random_state=42
)

In [ ]:
def __getitem__(self, idx):
    try:
        img = Image.open(self.image_paths[idx]).convert("RGB")
    except:
        return self.__getitem__((idx + 1) % len(self.image_paths))

    if self.transform:
        img = self.transform(img)

    return img, self.labels[idx]

In [ ]:
print("Final dataset size:", len(df))

In [ ]:
image_paths = df["image_path"].values
labels = df["label_encoded"].values

In [ ]:

print("min:", labels.min())
print("max:", labels.max())
#print("classes:",len(labels))

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
print("num_classes:", df["label_encoded"].nunique())
print("min label:", df["label_encoded"].min())
print("max label:", df["label_encoded"].max())

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class LandmarkDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
train_dataset = LandmarkDataset(
    train_df["image_path"].values,
    train_df["label_encoded"].values,
    transform=train_transforms
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
from torchvision import models
import torch.nn as nn

embedding_model = models.resnet50(pretrained=True)   # 🔥 upgrade
embedding_model.fc = nn.Identity()

embedding_model = embedding_model.to(device)
embedding_model.eval()

In [ ]:
model2 = models.efficientnet_b0(pretrained=True)
model2.classifier = nn.Identity()
model2 = model2.to(device)
model2.eval()

In [ ]:
import torch.nn.functional as F

def get_embeddings(loader):
    embeddings = []
    labels = []

    with torch.no_grad():
        for images, batch_labels in loader:
            images = images.to(device)

            emb1 = embedding_model(images)
            emb2 = model2(images)

            emb1 = F.normalize(emb1, p=2, dim=1)
            emb2 = F.normalize(emb2, p=2, dim=1)

            emb = torch.cat([emb1, emb2], dim=1)

            embeddings.append(emb.cpu())
            labels.append(batch_labels)

    return torch.cat(embeddings), torch.cat(labels)

In [ ]:
train_embeddings, train_labels = get_embeddings(train_loader)
#exttracting embe

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

dimension = train_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(train_embeddings.numpy())

In [ ]:
def search(query_embedding, k=10):
    D, I = index.search(query_embedding, k)
    return D, I

In [ ]:
def get_single_embedding(image):
    image = train_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        emb1 = embedding_model(image)
        emb2 = model2(image)

        emb1 = F.normalize(emb1, p=2, dim=1)
        emb2 = F.normalize(emb2, p=2, dim=1)

        emb = torch.cat([emb1, emb2], dim=1)

    return emb.cpu().numpy()

In [ ]:
def predict_weighted(I, D, train_labels):
    top_k_labels = train_labels[I[0]]
    distances = D[0]

    label_score = {}

    for label, dist in zip(top_k_labels, distances):
        weight = np.exp(dist*5)  # 🔥 key change (inverse distance)

        if label in label_score:
            label_score[label] += weight
        else:
            label_score[label] = weight

    return max(label_score, key=label_score.get)

In [ ]:
import numpy as np
y_pred = []
y_true = []

for i in range(len(test_df)):
    img = Image.open(test_df["image_path"].iloc[i]).convert("RGB")

    query_embedding = get_single_embedding(img)
    D, I = search(query_embedding, k=15)

    final = predict_weighted(I, D, train_labels)

    y_pred.append(final)
    y_true.append(test_df["label_encoded"].iloc[i])

In [ ]:
print(y_true)
print(y_pred)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
acc = accuracy_score(y_true, y_pred)
print("FINAL Accuracy:", acc)

In [ ]:
sample = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/sample_submission.csv")
test_ids = sample["id"].values
def get_test_path(image_id):
    return f"/kaggle/input/landmark-recognition-2021/test/{image_id[0]}/{image_id[1]}/{image_id[2]}/{image_id}.jpg"

test_image_paths = [get_test_path(i) for i in test_ids]

print("Total test images:", len(test_ids))

In [ ]:
def get_test_path(image_id):
    return f"/kaggle/input/landmark-recognition-2021/test/{image_id[0]}/{image_id[1]}/{image_id[2]}/{image_id}.jpg"

In [ ]:
sample = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/sample_submission.csv")

# dummy prediction (structure check ke liye)
submission = sample.copy()

submission["landmarks"] = "0 0.0"   # temporary

submission.to_csv("submission.csv", index=False)